In [1]:
import os
import pandas as pd

In [2]:
# Paths
csv_path = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/sls_Daily/'
masterlist = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/ALL_STATIONS_FINAL_2_00.csv' # same for monthly and daily
output_path = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/CSV/DAILY_CSV_COMBINED/'

os.makedirs(output_path, exist_ok=True)

# Load masterlist
stations_df = pd.read_csv(masterlist)
station_num = stations_df['grdc_no'].unique().tolist()
print(len(station_num))

578


In [3]:
import os
import pandas as pd
import numpy as np

# Folder with all individual combined CSVs
csv_path = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/CSV/DAILY_CSV_COMBINED/'
output_file = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/ALL_STATIONS_ALL_DAYS_COMBINED_2.csv'

# ---- USER-DEFINED FILTERS ----
Q_THRESHOLD = 1.0          # Q values below this are considered "low"
MIN_FRACTION_BELOW = 0.9   # exclude if >70% of Q values are below threshold
# ------------------------------

# List all CSV files in the folder
csv_files = sorted([f for f in os.listdir(csv_path) if f.endswith('.csv')])

print(f"Found {len(csv_files)} CSV files to process.\n")

df_list = []
excluded_files = []

for i, file in enumerate(csv_files, 1):
    file_path = os.path.join(csv_path, file)

    try:
        df = pd.read_csv(file_path)

        if 'Q' not in df.columns:
            print(f"⚠️ [{i}/{len(csv_files)}] {file} skipped (no 'Q' column)")
            excluded_files.append(file)
            continue

        # Drop NaNs before computing fraction
        q_vals = df['Q'].dropna()

        if len(q_vals) == 0:
            print(f"⚠️ [{i}/{len(csv_files)}] {file} skipped (all Q are NaN)")
            excluded_files.append(file)
            continue

        frac_below = (q_vals < Q_THRESHOLD).mean()

        if frac_below > MIN_FRACTION_BELOW:
            print(
                f"🚫 [{i}/{len(csv_files)}] Excluded {file} "
                f"({frac_below:.1%} of Q < {Q_THRESHOLD})"
            )
            excluded_files.append(file)
        else:
            df_list.append(df)
            print(
                f"✅ [{i}/{len(csv_files)}] Loaded {file} "
                f"({len(df)} rows, {frac_below:.1%} of Q < {Q_THRESHOLD})"
            )

    except Exception as e:
        print(f"❌ Error loading {file}: {e}")
        excluded_files.append(file)

# Combine all retained dataframes
if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    print(f"\n📊 Combined DataFrame shape: {combined_df.shape}")

    combined_df.to_csv(output_file, index=False)
    print(f"💾 Saved combined CSV to:\n{output_file}")
else:
    print("\n⚠️ No dataframes passed the filtering criteria.")

# Summary
print("\n📌 Summary")
print(f"   Total files found:     {len(csv_files)}")
print(f"   Files included:        {len(df_list)}")
print(f"   Files excluded:        {len(excluded_files)}")

Found 574 CSV files to process.

✅ [1/574] Loaded 1159100_combined.csv (2363 rows, 2.1% of Q < 1.0)
✅ [2/574] Loaded 1159103_combined.csv (2365 rows, 2.1% of Q < 1.0)
✅ [3/574] Loaded 1159105_combined.csv (2364 rows, 52.9% of Q < 1.0)
🚫 [4/574] Excluded 1159125_combined.csv (98.0% of Q < 1.0)
✅ [5/574] Loaded 1159130_combined.csv (2355 rows, 71.6% of Q < 1.0)
✅ [6/574] Loaded 1159290_combined.csv (2392 rows, 30.9% of Q < 1.0)
✅ [7/574] Loaded 1159300_combined.csv (2392 rows, 13.7% of Q < 1.0)
✅ [8/574] Loaded 1159301_combined.csv (2327 rows, 37.8% of Q < 1.0)
✅ [9/574] Loaded 1159302_combined.csv (2391 rows, 0.7% of Q < 1.0)
✅ [10/574] Loaded 1159304_combined.csv (2391 rows, 6.9% of Q < 1.0)
🚫 [11/574] Excluded 1159305_combined.csv (94.9% of Q < 1.0)
✅ [12/574] Loaded 1159330_combined.csv (2379 rows, 2.4% of Q < 1.0)
✅ [13/574] Loaded 1159335_combined.csv (2379 rows, 68.8% of Q < 1.0)
✅ [14/574] Loaded 1159400_combined.csv (2286 rows, 88.1% of Q < 1.0)
✅ [15/574] Loaded 1159403_combine

In [3]:
# Column order you want in final files
final_order = [
    'Date','grdc_no', 'river', 'station', 'country', 'lat', 'long', 'area', 'area_delin',
    'altitude', 'Q', 'SWE', 'SWE_scaled', 'r2', 'P',
    'avg_slope', 'max_slope',
    *[f'LC_{i}' for i in range(1, 21)],
    *[f'ST_{i}' for i in range(1, 17)]
]

# Loop over all stations
for _, row in stations_df.iterrows():
    num = int(row['grdc_no'])
    csv_file = os.path.join(csv_path, f"{num}_parallel.csv")
    
    if not os.path.exists(csv_file):
        candidates = [
            os.path.join(csv_path, f"{num}_parallel_rerun.csv"),
            os.path.join(csv_path, f"{num}_parallel2.csv")
        ]
    
        loaded = False
        for candidate in candidates:
            if os.path.exists(candidate):
                csv_file = candidate
                loaded = True
                break
    
        if not loaded:
            print(f"⚠️ File not found: {num}")
            continue

    try:
        # Read station monthly data
        df = pd.read_csv(csv_file)

        # Drop index column if present
        if 'Unnamed: 0' in df.columns:
            df = df.drop(columns=['Unnamed: 0'])

        # Add metadata from masterlist to each row
        for col in stations_df.columns:
            if col in row and col not in df.columns:
                df[col] = row[col]

        # Ensure all final columns exist (fill missing with NaN)
        for col in final_order:
            if col not in df.columns:
                df[col] = pd.NA

        # Reorder columns according to final_order
        df = df[final_order]
        # print(df.head())
        # Save to output directory
        out_file = os.path.join(output_path, f"{num}_combined.csv")
        df.to_csv(out_file, index=False)
        print(f"✅ Saved {num}_combined.csv ({len(df)} rows)")

    except Exception as e:
        print(f"❌ Error processing {num}: {e}")

print("\n🎉 All files processed and reordered successfully!")

⚠️ File not found: 1159100
⚠️ File not found: 1159103
✅ Saved 1159105_combined.csv (2364 rows)
✅ Saved 1159125_combined.csv (1733 rows)
✅ Saved 1159130_combined.csv (2355 rows)
✅ Saved 1159290_combined.csv (2392 rows)
✅ Saved 1159300_combined.csv (2392 rows)
✅ Saved 1159301_combined.csv (2327 rows)
✅ Saved 1159302_combined.csv (2391 rows)
✅ Saved 1159304_combined.csv (2391 rows)
✅ Saved 1159305_combined.csv (1950 rows)
✅ Saved 1159330_combined.csv (2379 rows)
✅ Saved 1159335_combined.csv (2379 rows)
✅ Saved 1159400_combined.csv (2286 rows)
✅ Saved 1159403_combined.csv (1901 rows)
✅ Saved 1159405_combined.csv (2286 rows)
✅ Saved 1159413_combined.csv (2267 rows)
✅ Saved 1159415_combined.csv (1748 rows)
✅ Saved 1159417_combined.csv (1930 rows)
✅ Saved 1159450_combined.csv (2286 rows)
✅ Saved 1159490_combined.csv (2289 rows)
✅ Saved 1159510_combined.csv (1820 rows)
✅ Saved 1159525_combined.csv (1869 rows)
✅ Saved 1159535_combined.csv (2245 rows)
✅ Saved 1159545_combined.csv (2338 rows)
✅ S

In [4]:
# Folder with all individual combined CSVs
csv_path = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/CSV/DAILY_CSV_COMBINED/'
output_file = '/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/ALL_STATIONS_ALL_DAYS_COMBINED.csv'

# List all CSV files in the folder
csv_files = sorted([f for f in os.listdir(csv_path) if f.endswith('.csv')])

print(f"Found {len(csv_files)} CSV files to combine.\n")

# List to hold dataframes
df_list = []

for i, file in enumerate(csv_files, 1):
    file_path = os.path.join(csv_path, file)
    try:
        df = pd.read_csv(file_path)
        df_list.append(df)
        print(f"✅ [{i}/{len(csv_files)}] Loaded {file} ({len(df)} rows)")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")

# Combine all into one big dataframe
if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    print(f"\n📊 Combined DataFrame shape: {combined_df.shape}")

    # Save combined output
    combined_df.to_csv(output_file, index=False)
    print(f"💾 Saved combined CSV to:\n{output_file}")
else:
    print("⚠️ No dataframes loaded — check your input folder path.")

Found 574 CSV files to combine.

✅ [1/574] Loaded 1159100_combined.csv (2363 rows)
✅ [2/574] Loaded 1159103_combined.csv (2365 rows)
✅ [3/574] Loaded 1159105_combined.csv (2364 rows)
✅ [4/574] Loaded 1159125_combined.csv (1733 rows)
✅ [5/574] Loaded 1159130_combined.csv (2355 rows)
✅ [6/574] Loaded 1159290_combined.csv (2392 rows)
✅ [7/574] Loaded 1159300_combined.csv (2392 rows)
✅ [8/574] Loaded 1159301_combined.csv (2327 rows)
✅ [9/574] Loaded 1159302_combined.csv (2391 rows)
✅ [10/574] Loaded 1159304_combined.csv (2391 rows)
✅ [11/574] Loaded 1159305_combined.csv (1950 rows)
✅ [12/574] Loaded 1159330_combined.csv (2379 rows)
✅ [13/574] Loaded 1159335_combined.csv (2379 rows)
✅ [14/574] Loaded 1159400_combined.csv (2286 rows)
✅ [15/574] Loaded 1159403_combined.csv (1901 rows)
✅ [16/574] Loaded 1159405_combined.csv (2286 rows)
✅ [17/574] Loaded 1159413_combined.csv (2267 rows)
✅ [18/574] Loaded 1159415_combined.csv (1748 rows)
✅ [19/574] Loaded 1159417_combined.csv (1930 rows)
✅ [20/5

/tmp/ipykernel_2822900/846985978.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(df_list, ignore_index=True)



📊 Combined DataFrame shape: (892798, 53)
💾 Saved combined CSV to:
/global/scratch/users/cgerlein/fc_ecohydrology_scratch/CSMUB/RESULTS/ALL_STATIONS_ALL_DAYS_COMBINED.csv
